In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import gc

# ==========================================
# 0. CONFIGURATION & DATA LOADING
# ==========================================
train_path = "/kaggle/input/competitions/ts-forecasting/train.parquet"
test_path = "/kaggle/input/competitions/ts-forecasting/test.parquet"

train_full = pd.read_parquet(train_path)
test_full = pd.read_parquet(test_path)

TARGET = "y_target"
FORECAST_WINDOWS = [1, 3, 10, 25]
N_FOLDS = 10

lgb_cfg = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'n_estimators': 1500, # Bumped up slightly since we rely purely on trees now
    'num_leaves': 63,
    'min_child_samples': 100,
    'feature_fraction': 0.7,
    'verbosity': -1,
    'n_jobs': -1,
    'random_state': 42
}

# ==========================================
# 1. HELPERS & METRICS
# ==========================================
def weighted_rmse_score(y_true, y_pred, w):
    """Custom metric for the competition."""
    if w is None:
        w = np.ones_like(y_true)
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true ** 2)) + 1e-8
    return float(np.sqrt(num / den))

# ==========================================
# 2. 10-FOLD OOF PIPELINE
# ==========================================
test_outputs = []
overall_oof_y = []
overall_oof_pred = []
overall_oof_weights = []

for horizon in FORECAST_WINDOWS:
    print(f"\n{'='*40}")
    print(f" FORECAST HORIZON: {horizon}")
    print(f"{'='*40}")
    
    tr_df = train_full[train_full['horizon'] == horizon].reset_index(drop=True)
    te_df = test_full[test_full['horizon'] == horizon].reset_index(drop=True)

    if len(tr_df) == 0 or len(te_df) == 0:
        continue

    # Identify features
    drop_cols = [TARGET, 'ts_index', 'horizon']
    common_cols = list(set(tr_df.columns) & set(te_df.columns))
    features = sorted([c for c in common_cols if c not in drop_cols])

    X_train = tr_df[features].copy()
    y_train = tr_df[TARGET]
    X_test = te_df[features].copy()
    weights = tr_df['weight'].values if 'weight' in tr_df.columns else None

    # Handle Categoricals natively for LightGBM
    for col in features:
        if X_train[col].dtype == 'object':
            X_train[col] = X_train[col].astype('category')
            X_test[col] = X_test[col].astype('category')

    # Arrays to store blended test predictions and OOF predictions
    test_preds_blend = np.zeros(len(X_test))
    oof_preds = np.zeros(len(X_train))
    
    # 10-Fold Cross Validation Setup
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    
    horizon_oof_scores = []
    
    for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train)):
        # Split Data
        X_trn, y_trn = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        w_trn = weights[trn_idx] if weights is not None else None
        w_val = weights[val_idx] if weights is not None else None

        # Train Model
        model = lgb.LGBMRegressor(**lgb_cfg)
        model.fit(
            X_trn, y_trn,
            sample_weight=w_trn,
            eval_set=[(X_val, y_val)],
            eval_sample_weight=[w_val] if w_val is not None else None,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )
        
        # In-Fold Prediction & Score
        pred_trn = model.predict(X_trn)
        score_trn = weighted_rmse_score(y_trn.values, pred_trn, w_trn)
        
        # Out-Of-Fold Prediction & Score
        pred_val = model.predict(X_val)
        score_val = weighted_rmse_score(y_val.values, pred_val, w_val)
        
        oof_preds[val_idx] = pred_val
        horizon_oof_scores.append(score_val)
        
        # Accumulate Test Predictions (Averaging)
        test_preds_blend += model.predict(X_test) / N_FOLDS
        
        print(f"Fold {fold+1:02d} | In-Fold WRMSE: {score_trn:.5f} | OOF WRMSE: {score_val:.5f} | Trees: {model.best_iteration_}")

    # Horizon Summary
    horizon_w_val = weights if weights is not None else np.ones_like(y_train)
    horizon_total_score = weighted_rmse_score(y_train.values, oof_preds, horizon_w_val)
    print(f"\n>>> Horizon {horizon} Overall OOF WRMSE: {horizon_total_score:.5f}")
    
    # Store for global tracking
    overall_oof_y.extend(y_train.values)
    overall_oof_pred.extend(oof_preds)
    overall_oof_weights.extend(horizon_w_val)

    # Save Test Predictions
    te_df['prediction'] = test_preds_blend
    test_outputs.append(te_df[['id', 'prediction']])

    # Cleanup memory before next horizon
    del tr_df, te_df, X_train, X_test, y_train
    gc.collect()

# ==========================================
# 3. FINAL EVALUATION & SUBMISSION
# ==========================================
final_global_score = weighted_rmse_score(
    np.array(overall_oof_y),
    np.array(overall_oof_pred),
    np.array(overall_oof_weights)
)

print(f"\n{'='*40}")
print(f"🏆 FINAL GLOBAL OOF WRMSE: {final_global_score:.5f}")
print(f"{'='*40}\n")

submission = pd.concat(test_outputs, ignore_index=True)
submission = submission[['id', 'prediction']]
submission.dropna(subset=['prediction'], inplace=True)
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv("submission.csv", index=False)

print("✅ submission.csv successfully generated and formatted.")
